In [ ]:
# scripts/09_hyperparameter_optimization.py

"""
Step 09 - Build functional space (rolling windows) and optimize functional k-NN hyperparameters (EXTENDED metric).
- Builds the functional space F from a wide-by-year dataset (e.g., VAR_2019 ... VAR_2023).
- Creates RQ_final using each firm's last available year.
- Optimizes (k, lambda) and then weights using Optuna (F1-score as objective).
- Extended distance includes:
  - bounded functional distance (financial trajectories)
  - categorical distance for DEP
  - categorical distance for CIIU_Letra
  - normalized time-lag distance using Año_final
- Saves:
  - outputs/data/functional_space.parquet
  - outputs/models/functional_knn_params.pkl
  - outputs/reports/step09_summary.txt
  - outputs/reports/step09_top3.csv
"""

from __future__ import annotations

import re
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score


# ---------------------------------------------------------------------
# Step 1: Configuration
# ---------------------------------------------------------------------
SEED_K_SAMPLE = 123
SEED_W_SAMPLE = 222
SEED_CV = 42

# Repo-relative paths
INPUT_PATH = Path("Data") / "functional_space_base.parquet"
OUTPUT_SPACEF_PATH = Path("outputs") / "data" / "functional_space.parquet"
OUTPUT_PARAMS_PATH = Path("outputs") / "models" / "functional_knn_params.pkl"
OUTPUT_SUMMARY_PATH = Path("outputs") / "reports" / "step09_summary.txt"
OUTPUT_TOP3_CSV_PATH = Path("outputs") / "reports" / "step09_top3.csv"

N_WINDOW = 5
YEAR_MAX = 2023

SAMPLE_FRAC_OPTUNA_K = 0.10
SAMPLE_FRAC_OPTUNA_WEIGHTS = 0.10
N_TRIALS_OPTUNA_K = 30
N_TRIALS_OPTUNA_WEIGHTS = 30

VALID_YEARS = list(range(YEAR_MAX - 24, YEAR_MAX + 1))  # 25 years back (e.g., 1999..2023)
YEAR_MIN = min(VALID_YEARS)
MAX_YEAR_GAP = max(1, YEAR_MAX - YEAR_MIN)  # used to normalize time-lag to [0,1]


# Ensure output folders exist
for p in [OUTPUT_SPACEF_PATH, OUTPUT_PARAMS_PATH, OUTPUT_SUMMARY_PATH, OUTPUT_TOP3_CSV_PATH]:
    p.parent.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Step 2: Load base dataset
# ---------------------------------------------------------------------
base = pd.read_parquet(INPUT_PATH).drop_duplicates()
print(f"[OK] Base loaded: {base.shape}")


# ---------------------------------------------------------------------
# Step 3: Detect last available year per firm
# ---------------------------------------------------------------------
year_cols = [c for c in base.columns if re.match(r".+_\d{4}$", c)]
df_years = base[year_cols].copy()

last_years = []
for _, row in tqdm(df_years.iterrows(), total=df_years.shape[0], desc="Detecting last year per firm"):
    years = [
        int(col.split("_")[-1])
        for col in row.index
        if pd.notna(row[col]) and int(col.split("_")[-1]) in VALID_YEARS
    ]
    last_years.append(max(years) if years else None)

s_last_year = pd.Series(last_years, index=base.index)
valid_firms = base[s_last_year.notna()].copy()
s_last_year = s_last_year.dropna().astype(int)

print(f"[OK] Firms with valid last year: {valid_firms.shape[0]:,}")


# ---------------------------------------------------------------------
# Step 4: Build RQ_final using each firm's last available year
# ---------------------------------------------------------------------
rq_final = []
for idx, row in valid_firms.iterrows():
    y_end = s_last_year.loc[idx]
    rq_col = f"RQ_{y_end}"
    rq_val = row[rq_col] if (rq_col in row.index and pd.notna(row[rq_col])) else np.nan
    rq_final.append(rq_val)

valid_firms["RQ_final"] = rq_final
valid_firms = valid_firms[valid_firms["RQ_final"].notna()].copy()
valid_firms["RQ_final"] = valid_firms["RQ_final"].astype(int)

print(f"[OK] Firms with assigned RQ_final: {valid_firms.shape[0]:,}")


# ---------------------------------------------------------------------
# Step 5: Build functional space F with rolling windows
# ---------------------------------------------------------------------
year_cols = [c for c in valid_firms.columns if re.match(r".+_\d{4}$", c)]
fin_indicators = sorted({c.split("_")[0] for c in year_cols if not c.startswith("RQ")})

# Extra columns to keep (if present) for the extended metric
extra_cols = ["DEP", "CIIU_Letra"]

rows = []
for idx, row in tqdm(valid_firms.iterrows(), total=valid_firms.shape[0], desc="Building functional trajectories"):
    y_end = s_last_year.loc[idx]
    window_years = list(range(y_end - N_WINDOW + 1, y_end + 1))

    new_row = {
        "NIT": idx,
        "RQ_final": row["RQ_final"],
        "Año_final": y_end,  # keep this name (matches your downstream scripts)
    }

    for c in extra_cols:
        if c in row.index:
            new_row[c] = row[c]

    # Add functional trajectories (older -> newer: -4 ... -0)
    for var in fin_indicators:
        for i, year in enumerate(window_years[::-1]):
            original = f"{var}_{year}"
            new_name = f"{var}_-{N_WINDOW - 1 - i}"
            new_row[new_name] = row.get(original, np.nan)

    rows.append(new_row)

spaceF = pd.DataFrame(rows).set_index("NIT")
print(f"[OK] Functional space generated: {spaceF.shape}")


# ---------------------------------------------------------------------
# Step 6: Save functional space
# ---------------------------------------------------------------------
spaceF.to_parquet(OUTPUT_SPACEF_PATH)
print(f"[SAVED] Functional space: {OUTPUT_SPACEF_PATH}")


# ---------------------------------------------------------------------
# Step 7: Clean and prepare for optimization
# ---------------------------------------------------------------------
df = spaceF.copy()
df = df[df["RQ_final"].notna()].copy()

functional_cols = [c for c in df.columns if re.match(r".+_-\d$", c)]
fin_indicators = sorted({c.split("_")[0] for c in functional_cols})

# Ensure required extra columns exist (for extended metric).
# If they are missing, we create them with NaN so code doesn't crash,
# but in practice you WANT them present in the base dataset.
for c in ["DEP", "CIIU_Letra", "Año_final"]:
    if c not in df.columns:
        df[c] = np.nan


# ---------------------------------------------------------------------
# Distance functions (EXTENDED)
# ---------------------------------------------------------------------
def _bounded_indicator_distance(f1: pd.Series, f2: pd.Series, var: str, lambda_p: float, n: int) -> float:
    v1 = [f1.get(f"{var}_-{i}", np.nan) for i in range(n)]
    v2 = [f2.get(f"{var}_-{i}", np.nan) for i in range(n)]

    l1, valid = 0.0, 0
    for a, b in zip(v1, v2):
        if pd.notna(a) and pd.notna(b):
            if np.isinf(a) and np.isinf(b) and a == b:
                l1 += 0.0
            elif np.isinf(a) or np.isinf(b):
                l1 += np.inf
            else:
                l1 += abs(a - b)
            valid += 1

    missing = n - valid
    if valid > 0:
        penalized = l1 * (1 + lambda_p * (missing / n))
        bounded = penalized / (1 + penalized) if np.isfinite(penalized) else 1.0
    else:
        bounded = 1.0

    return float(bounded)


def extended_distance_unweighted(
    f1: pd.Series, f2: pd.Series, lambda_p: float, n: int = N_WINDOW,
    w_dep: float = 1.0, w_ciiu: float = 1.0, w_year: float = 1.0
) -> float:
    """
    Used in k/lambda optimization.
    Financial part is the mean bounded distance across indicators (unweighted),
    plus extra components (DEP, CIIU, normalized year gap) with fixed weights.
    """
    # Financial part
    total = 0.0
    for var in fin_indicators:
        total += _bounded_indicator_distance(f1, f2, var, lambda_p, n)

    d_fin = total / len(fin_indicators) if fin_indicators else 1.0

    # Extra components
    d_dep = 0.0 if f1.get("DEP", np.nan) == f2.get("DEP", np.nan) else 1.0
    d_ciiu = 0.0 if f1.get("CIIU_Letra", np.nan) == f2.get("CIIU_Letra", np.nan) else 1.0

    y1 = f1.get("Año_final", np.nan)
    y2 = f2.get("Año_final", np.nan)
    if pd.notna(y1) and pd.notna(y2):
        d_year = abs(float(y1) - float(y2)) / MAX_YEAR_GAP  # scaled to [0,1]
    else:
        d_year = 1.0  # penalize missing year info

    return float(d_fin + w_dep * d_dep + w_ciiu * d_ciiu + w_year * d_year)


def extended_distance_weighted(
    f1: pd.Series, f2: pd.Series, lambda_p: float, n: int,
    fin_weights: dict[str, float],
    w_dep: float, w_ciiu: float, w_year: float
) -> float:
    """
    Used in weight optimization.
    Financial part is weighted average of bounded distances, plus extra components.
    The final distance is normalized by total weight mass.
    """
    total, wsum = 0.0, 0.0

    for var in fin_indicators:
        d_var = _bounded_indicator_distance(f1, f2, var, lambda_p, n)
        w = float(fin_weights.get(var, 1.0))
        total += w * d_var
        wsum += w

    d_dep = 0.0 if f1.get("DEP", np.nan) == f2.get("DEP", np.nan) else 1.0
    d_ciiu = 0.0 if f1.get("CIIU_Letra", np.nan) == f2.get("CIIU_Letra", np.nan) else 1.0

    y1 = f1.get("Año_final", np.nan)
    y2 = f2.get("Año_final", np.nan)
    if pd.notna(y1) and pd.notna(y2):
        d_year = abs(float(y1) - float(y2)) / MAX_YEAR_GAP
    else:
        d_year = 1.0

    total += w_dep * d_dep + w_ciiu * d_ciiu + w_year * d_year
    wsum += float(w_dep + w_ciiu + w_year)

    return float(total / wsum) if wsum > 0 else 1.0


# ---------------------------------------------------------------------
# Step 8: Optimize k and lambda (Optuna) using EXTENDED distance
# ---------------------------------------------------------------------
m_k = df.sample(frac=SAMPLE_FRAC_OPTUNA_K, random_state=SEED_K_SAMPLE)
X_k = m_k[functional_cols + ["DEP", "CIIU_Letra", "Año_final"]].copy()
y_k = m_k["RQ_final"].copy()


def objective_k_lambda(trial: optuna.Trial) -> float:
    k = trial.suggest_int("k", 3, 15)
    lambda_p = trial.suggest_float("lambda", 0.1, 5.0)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED_CV)
    f1s = []

    for train_idx, test_idx in skf.split(X_k, y_k):
        X_train, X_test = X_k.iloc[train_idx], X_k.iloc[test_idx]
        y_train, y_test = y_k.iloc[train_idx], y_k.iloc[test_idx]

        preds = []
        for _, row_test in X_test.iterrows():
            dists = [
                (extended_distance_unweighted(row_test, X_train.iloc[j], lambda_p, n=N_WINDOW), y_train.iloc[j])
                for j in range(len(X_train))
            ]
            neighbors = sorted(dists, key=lambda x: x[0])[:k]
            pred = round(np.mean([v[1] for v in neighbors]))
            preds.append(pred)

        f1s.append(f1_score(y_test, preds))

    return float(np.mean(f1s))


print("[RUN] Optimizing k and lambda with Optuna (extended metric)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)

study_k = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED_CV),
)

for _ in tqdm(range(N_TRIALS_OPTUNA_K), desc="Searching k & lambda"):
    study_k.optimize(objective_k_lambda, n_trials=1, catch=(Exception,))

k_opt = int(study_k.best_params["k"])
lambda_opt = float(study_k.best_params["lambda"])
best_f1_k = float(study_k.best_value)

print(f"[OK] Best k: {k_opt}, best lambda: {lambda_opt:.4f}, F1: {best_f1_k:.4f}")


# ---------------------------------------------------------------------
# Step 9: Optimize weights (Optuna) using EXTENDED distance
# ---------------------------------------------------------------------
m_w = df.sample(frac=SAMPLE_FRAC_OPTUNA_WEIGHTS, random_state=SEED_W_SAMPLE)
X_w = m_w[functional_cols + ["DEP", "CIIU_Letra", "Año_final"]].copy()
y_w = m_w["RQ_final"].copy()


def objective_weights(trial: optuna.Trial) -> float:
    # Financial weights
    fin_weights = {var: trial.suggest_float(f"weight_{var}", 0.1, 5.0) for var in fin_indicators}

    # Extra weights
    w_dep = trial.suggest_float("weight_DEP", 0.1, 5.0)
    w_ciiu = trial.suggest_float("weight_CIIU", 0.1, 5.0)
    w_year = trial.suggest_float("weight_year", 0.1, 5.0)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED_CV)
    f1s = []

    for train_idx, test_idx in skf.split(X_w, y_w):
        X_train, X_test = X_w.iloc[train_idx], X_w.iloc[test_idx]
        y_train, y_test = y_w.iloc[train_idx], y_w.iloc[test_idx]

        preds = []
        for _, row_test in X_test.iterrows():
            dists = [
                (
                    extended_distance_weighted(
                        row_test, X_train.iloc[j], lambda_opt, N_WINDOW,
                        fin_weights=fin_weights, w_dep=w_dep, w_ciiu=w_ciiu, w_year=w_year
                    ),
                    y_train.iloc[j],
                )
                for j in range(len(X_train))
            ]
            neighbors = sorted(dists, key=lambda x: x[0])[:k_opt]
            pred = round(np.mean([v[1] for v in neighbors]))
            preds.append(pred)

        f1s.append(f1_score(y_test, preds))

    return float(np.mean(f1s))


print("[RUN] Optimizing weights with Optuna (extended metric)...")
study_w = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED_CV),
)

for _ in tqdm(range(N_TRIALS_OPTUNA_WEIGHTS), desc="Searching weights"):
    study_w.optimize(objective_weights, n_trials=1, catch=(Exception,))

best_params = {k: float(v) for k, v in study_w.best_params.items()}

best_fin_weights = {k.replace("weight_", ""): v for k, v in best_params.items() if k.startswith("weight_") and k not in ["weight_DEP", "weight_CIIU", "weight_year"]}
w_dep_opt = best_params["weight_DEP"]
w_ciiu_opt = best_params["weight_CIIU"]
w_year_opt = best_params["weight_year"]

# Top-3 financial indicators only
top3 = sorted(best_fin_weights.items(), key=lambda x: -x[1])[:3]
best_f1_w = float(study_w.best_value)

print("\n[RESULT] Top 3 FINANCIAL indicators by weight:")
for var, w in top3:
    print(f" - {var}: {w:.4f}")

print("\n[RESULT] Extra component weights:")
print(f" - DEP:  {w_dep_opt:.4f}")
print(f" - CIIU: {w_ciiu_opt:.4f}")
print(f" - Year: {w_year_opt:.4f}")

print(f"\n[OK] Final F1 with optimal weights: {best_f1_w:.4f}")


# ---------------------------------------------------------------------
# Step 10: Save optimal parameters + summary report
# ---------------------------------------------------------------------
params = {
    "k": k_opt,
    "lambda": lambda_opt,

    # Financial weights
    "weights": best_fin_weights,
    "top3": top3,

    # Extra weights (extended metric)
    "weight_DEP": w_dep_opt,
    "weight_CIIU": w_ciiu_opt,
    "weight_year": w_year_opt,

    # Optimization diagnostics
    "f1_k_lambda": best_f1_k,
    "f1_final": best_f1_w,

    # Metadata
    "n_window": N_WINDOW,
    "year_max": YEAR_MAX,
    "year_min": YEAR_MIN,
    "max_year_gap": MAX_YEAR_GAP,
    "valid_years": VALID_YEARS,
    "sample_frac_k": SAMPLE_FRAC_OPTUNA_K,
    "sample_frac_weights": SAMPLE_FRAC_OPTUNA_WEIGHTS,
    "n_trials_k": N_TRIALS_OPTUNA_K,
    "n_trials_weights": N_TRIALS_OPTUNA_WEIGHTS,
}

with open(OUTPUT_PARAMS_PATH, "wb") as f:
    pickle.dump(params, f)

print(f"[SAVED] Optimal parameters: {OUTPUT_PARAMS_PATH}")

with open(OUTPUT_SUMMARY_PATH, "w", encoding="utf-8") as f:
    f.write("Functional k-NN - Step 09 Summary (Extended Metric)\n")
    f.write("============================================================\n\n")

    f.write("Functional space construction\n")
    f.write(f" - Output file: {OUTPUT_SPACEF_PATH.as_posix()}\n")
    f.write(f" - Firms (rows): {spaceF.shape[0]:,}\n")
    f.write(f" - Functional columns: {len(functional_cols)}\n")
    f.write(" - Extra columns kept (if available): DEP, CIIU_Letra, Año_final\n\n")

    f.write("Hyperparameter optimization (Optuna)\n")
    f.write(f" - Best k: {k_opt}\n")
    f.write(f" - Best lambda: {lambda_opt:.4f}\n")
    f.write(f" - Best F1 (k, lambda stage): {best_f1_k:.4f}\n\n")

    f.write("Weight optimization (Optuna)\n")
    f.write(" - Top 3 financial indicators:\n")
    for var, w in top3:
        f.write(f"   * {var}: {w:.4f}\n")
    f.write("\n - Extra weights:\n")
    f.write(f"   * DEP:  {w_dep_opt:.4f}\n")
    f.write(f"   * CIIU: {w_ciiu_opt:.4f}\n")
    f.write(f"   * Year (normalized gap): {w_year_opt:.4f}\n")
    f.write(f"\n - Final F1 with optimal weights: {best_f1_w:.4f}\n\n")

    f.write("Time-lag normalization\n")
    f.write(f" - YEAR_MIN: {YEAR_MIN}\n")
    f.write(f" - YEAR_MAX: {YEAR_MAX}\n")
    f.write(f" - MAX_YEAR_GAP: {MAX_YEAR_GAP}\n\n")

    f.write(f"Parameters saved at: {OUTPUT_PARAMS_PATH.as_posix()}\n")

print(f"[SAVED] Summary report: {OUTPUT_SUMMARY_PATH}")

pd.DataFrame(top3, columns=["indicator", "weight"]).to_csv(OUTPUT_TOP3_CSV_PATH, index=False)
print(f"[SAVED] Top3 CSV: {OUTPUT_TOP3_CSV_PATH}")